## 1. Prepare all Nigeria H3 Cells  at Resolution 7

### 00. Step Up

In [ ]:
%load_ext autoreload
%autoreload 2 

import sys
from pathlib import Path 
import logging 
from datetime import datetime
import pickle
import h3
from codebase.utils.utils import setup_logging 
import sys 
from config.settings import STORAGE_CONFIG, ADMIN_DATA_SOURCES, INPUT_BASE_DATA_SOURCES, RAW_DATA_DIR,PROCESSED_DATA_DIR, EXPORTS_DIR, OUTPUT_DIR

import pandas as pd
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path'] 


logger = setup_logging(log_dir='log-main-sp-clustering-and-routing-res7', 
                       projname='log-main-spcr')

### 01. Load and Preprocess Boundary Data

In [45]:
%load_ext autoreload
%autoreload 2 


import sys
from pathlib import Path
project_root = Path('').parent
sys.path.append(str(project_root))


from pathlib import Path 
import logging 
from datetime import datetime
import pickle
from src.h3_spatial_system.data.downloader import DataDownloader, validate_and_summarize_data 
from src.h3_spatial_system.h3_system.generator import H3AddressGenerator, prepare_h3_address_as_dataframe
from src.h3_spatial_system.storage.duckdb_storage import DuckDBStorage
from config.settings import * 


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


validate_downloaded_data with the function validate_and_summarize_data save the standardized geojson file to disk

In [3]:
# Step 1: Download administrative boundary data
'''
logger.info("📥 Step 1: Downloading administrative boundary data...")
downloader = DataDownloader()
downloaded_files = downloader.download_admin_boundaries()
if downloaded_files:
    validation_results, summaries = validate_and_summarize_data(downloaded_files)
    
if not downloaded_files:
    logger.error("❌ Failed to download administrative boundary data")
    # return 1

logger.info(f"✅ Downloaded {len(downloaded_files)} boundary files")

'''

# load geojson files
dict_path_geojson ={
    'states': RAW_DATA_DIR / 'grid3-nga-operational-state-boundaries_standardized.geojson',
    'lgas': RAW_DATA_DIR / 'grid3-nga-operational-lga-boundaries_standardized.geojson',
    'wards': RAW_DATA_DIR / 'grid3-nga-operational-wards-v1-0_standardized.geojson'
}

logger.info("🚀 Starting H3-based address system generation for Nigeria")

# Step 2: Initialize H3 address generator
logger.info("🔧 Step 2: Initializing H3 address generator...")
generator = H3AddressGenerator(resolution=7)

2025-08-09 08:59:03,594 - INFO - 🚀 Starting H3-based address system generation for Nigeria
INFO:log-main-spcr:🚀 Starting H3-based address system generation for Nigeria
2025-08-09 08:59:03,598 - INFO - 🔧 Step 2: Initializing H3 address generator...
INFO:log-main-spcr:🔧 Step 2: Initializing H3 address generator...


In [4]:
# Step 3: Load administrative boundaries
logger.info("🗺️ Step 3: Loading administrative boundaries...")
states_path = str(dict_path_geojson['states'])
lgas_path = str(dict_path_geojson['lgas'])
wards_path = str(dict_path_geojson['wards'])

generator.load_admin_boundaries(states_path, lgas_path, wards_path)
logger.info("✅ Administrative boundaries loaded")

2025-08-09 08:59:08,797 - INFO - 🗺️ Step 3: Loading administrative boundaries...
INFO:log-main-spcr:🗺️ Step 3: Loading administrative boundaries...
INFO:src.h3_spatial_system.h3_system.generator:Loading administrative boundaries...
INFO:src.h3_spatial_system.h3_system.admin_assignment:Loading administrative boundaries...
INFO:src.h3_spatial_system.h3_system.admin_assignment:Filtered out 102 invalid geometries from ward data
INFO:src.h3_spatial_system.h3_system.admin_assignment:Loaded 37 states, 774 LGAs, 9308 wards
INFO:src.h3_spatial_system.h3_system.admin_assignment:states: All 37 units have valid geometries
INFO:src.h3_spatial_system.h3_system.admin_assignment:lgas: All 774 units have valid geometries
INFO:src.h3_spatial_system.h3_system.admin_assignment:wards: All 9308 units have valid geometries
INFO:src.h3_spatial_system.h3_system.admin_assignment:Built spatial index for states: 37 units
INFO:src.h3_spatial_system.h3_system.admin_assignment:Built spatial index for lgas: 774 units

### 02. Generate H3 Cells

In [6]:
# Step 4: Generate H3 cells
logger.info("🔷 Step 4: Generating H3 cells...")
h3_cells = generator.generate_h3_cells() 
with open(PROCESSED_DATA_DIR / 'h3_cells_res7.pickle', 'wb') as f:
    pickle.dump(h3_cells, f)

2025-08-09 09:00:26,784 - INFO - 🔷 Step 4: Generating H3 cells...
INFO:log-main-spcr:🔷 Step 4: Generating H3 cells...
INFO:src.h3_spatial_system.h3_system.generator:Generating H3 cells at resolution 7...
INFO:src.h3_spatial_system.h3_system.generator:Generated 308,419 H3 cells


### 03. Add Meta data to H3 Cells

In [7]:
%load_ext autoreload
%autoreload 2 


import sys
from pathlib import Path
import logging 

import h3
import pickle
from src.h3_spatial_system.h3_system.utils import h3_to_objects_parallel_safe, h3_to_objects_parallel_generator, H3SQLiteManager

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
# Load Data
with open('./data/processed/h3_cells_res7.pickle', 'rb') as f:
    h3_cells = pickle.load(f)
logger.info(f"✅ Generated {len(h3_cells):,} H3 cells")

2025-08-09 09:01:17,379 - INFO - ✅ Generated 308,419 H3 cells
INFO:log-main-spcr:✅ Generated 308,419 H3 cells


In [ ]:
# import time 
# start_time = time.time()   
# h3_cells_processed =  h3_to_objects_parallel_safe(h3_cells[0:1000]) 
# parallel_time = time.time() - start_time 
# print(f"Total time taken {len(h3_cells):,} cells took: {parallel_time:.2f} seconds") 

In [ ]:
# # For very large datasets, use batched processing:
# h3_data = {}
# for h3_index, result in h3_to_objects_parallel_generator(h3_cells):
#     h3_data[h3_index] = result ## 9m 17s

# with open('./data/processed/h3_cells_processed_dict.pickle', 'wb') as f:
#     pickle.dump(h3_data, f)



# METHOD 2
# Stream directly to JSONL - most efficient
# Process and save your H3 data
# import time 
# start_time = time.time()   
# with H3SQLiteManager("./data/processed/h3_data.db") as db:
#     for h3_index, result in h3_to_objects_parallel_generator(h3_cells):
#         db.add_result(h3_index, result)

# parallel_time = time.time() - start_time 
# print(f"Total time taken {len(h3_cells):,} cells took: {parallel_time:.2f} seconds") 
# print("Done! Database saved with compression.")

In [10]:
from src.h3_spatial_system.h3_system.FastH3DuckDBManager import FastH3DuckDBManager
from src.h3_spatial_system.h3_system.save_h3_data_utils import verify_db_file_and_data    
from config.settings import STORAGE_CONFIG

H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']


# METHOD 3
# Stream directly to DUCKDB - most efficient
with FastH3DuckDBManager(resolution=7, db_path=H3_DUCKDB_PATH, batch_size=100000) as db:
    count = 0
    for h3_index, result in h3_to_objects_parallel_generator(h3_cells):
        db.add_result(h3_index, result)
        count += 1
        
        # Check database periodically to verify commits
        if count % 100000 == 0:
            stats = db.get_stats()
            print(f"Processed: {count:,}, In DB: {stats['total_records']:,}")

# # After the context manager exits, check final state
# verify_db_file_and_data(db_path)

🚀 Initializing FastH3DuckDBManager (batch_size: 100,000)
📋 Creating optimized h3_cells table...
✅ Optimized table structure created
⚠️  Setting failed: SET wal_autocheckpoint=50000 - Parser Error: Unknown unit for memory_limit: '' (expected: KB, MB, GB, TB for 1000^i units or KiB, MiB, GiB, TiB for 1024^i units)
⚠️  Setting failed: PRAGMA enable_verification=false - Catalog Error: unrecognized configuration parameter "enable_verification"

Did you mean: "enable_profiling"
⚠️  Setting failed: PRAGMA force_checkpoint=false - Catalog Error: unrecognized configuration parameter "force_checkpoint"

Did you mean: "debug_checkpoint_abort"
⚡ Applied 7/10 performance optimizations
✅ Fast H3DuckDBManager ready
⚡ Saved 100,000 items in 2.57s (38,945/s) | Total: 100,000 (2,427/s avg)
Processed: 100,000, In DB: 2,258,746
⚡ Saved 100,000 items in 2.08s (48,068/s) | Total: 200,000 (2,534/s avg)
Processed: 200,000, In DB: 2,358,746
⚡ Saved 100,000 items in 2.15s (46,457/s) | Total: 300,000 (2,644/s av

In [12]:
# Check if your current database file exists and has data
from src.h3_spatial_system.h3_system.save_h3_data_utils import verify_db_file_and_data
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']
# verify_db_file_and_data(db_path)
verify_db_file_and_data(H3_DUCKDB_PATH) # Total records: 2,158,746 # 📊 Total records: 2,467,165

🔍 DuckDB File & Data Verification
📁 Database path: /home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/exports/h3_data.duckdb
📁 Directory: /home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/exports
✅ File exists! Size: 1,207,185,408 bytes (1151.26 MB)
✅ Successfully connected to database
📋 Tables in database: ['h3_cells']
✅ h3_cells table found
📊 Table structure:
   - h3_index: VARCHAR
   - resolution: TINYINT
   - centroid_lat: DOUBLE
   - centroid_lng: DOUBLE
   - polygon_wkt: VARCHAR
   - boundary_json: VARCHAR
   - latlng_json: VARCHAR
   - polygon_area: DOUBLE
   - num_vertices: SMALLINT
   - error: VARCHAR
   - created_at: TIMESTAMP
   - h3_derived_id: VARCHAR
   - grid_position_id: VARCHAR
   - primary_address_id: VARCHAR
   - country_code: VARCHAR
   - country_name: VARCHAR
   - state_code: VARCHAR
   - state_name: VARCHAR
   - lga_code: VARCHAR
   - lga_name: VARCHAR
   - ward_code: VARCHAR
   - ward_name: VARCHAR
   - confidence_level: VAR

True

In [13]:
# print(len(h3_data))
# print(len(h3_data.keys()))

### 04. Generate Address and Address ID for the Cells & ADD TO DB

In [37]:
%load_ext autoreload
%autoreload 2 

from src.h3_spatial_system.h3_system.FastH3DuckDBManager import FastH3DuckDBManager
from src.h3_spatial_system.h3_system.save_h3_data_utils import verify_db_file_and_data   
from config.settings import STORAGE_CONFIG 
import time
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Step 5: Generate addresses - to-do optu
start_time = time.time() 
logger.info("🏠 Step 5: Generating addresses...") 
addresses = generator.generate_addresses(h3_cells) 
logger.info(f"✅ Generated {len(addresses)} address records") 
parallel_time = time.time() - start_time 
print(f"Total time taken {len(h3_cells):,} cells took: {parallel_time:.2f} seconds") 

# To be Implemented
# This can handle both new and existing H3 indices
# with FastH3DuckDBManager(resolution=8, db_path=db_path) as db:
    # address_data = generator.generate_addresses(h3_cells) 
    # db.upsert_address_data(address_data)

2025-08-09 09:49:31,092 - INFO - 🏠 Step 5: Generating addresses...
INFO:log-main-spcr:🏠 Step 5: Generating addresses...
INFO:src.h3_spatial_system.h3_system.generator:Generating addresses for 308419 H3 cells...
Processing H3 cells: 100%|██████████| 31/31 [07:59<00:00, 15.45s/it]
INFO:src.h3_spatial_system.h3_system.generator:Generated 308419 address records using sequential processing
2025-08-09 09:57:30,112 - INFO - ✅ Generated 308419 address records
INFO:log-main-spcr:✅ Generated 308419 address records


Total time taken 308,419 cells took: 479.02 seconds


In [ ]:
PATH_H3_ADDRESS_DF_RES7_PICKLE = EXPORTS_DIR / 'all_h3_res7_nigeria_addresses.pickle'
with open(PATH_H3_ADDRESS_DF_RES7_PICKLE, 'wb') as file:
    pickle.dump(addresses, file)

In [ ]:
PATH_DF_H3_ADDRESS_DF_RES7_ALL = EXPORTS_DIR / 'df_all_h3_res7_nigeria_addresses.parquet' 
PATH_DF_H3_ADDRESS_DF_RES7_FILTER = EXPORTS_DIR / 'df_filterd_h3_res7_nigeria_addresses.parquet' 

df_address = prepare_h3_address_as_dataframe(address_data=addresses)
# Save Address as dataframe: 
df_address.to_parquet(PATH_DF_H3_ADDRESS_DF_RES7_ALL)
df_address.query('confidence_level != "manual_review"').to_parquet(PATH_DF_H3_ADDRESS_DF_RES7_FILTER)

In [49]:
# Verifing DB
verify_db_file_and_data(H3_DUCKDB_PATH)

🔍 DuckDB File & Data Verification
📁 Database path: /home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/exports/h3_data.duckdb
📁 Directory: /home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/exports
✅ File exists! Size: 1,207,185,408 bytes (1151.26 MB)
✅ Successfully connected to database
📋 Tables in database: ['h3_cells']
✅ h3_cells table found
📊 Table structure:
   - h3_index: VARCHAR
   - resolution: TINYINT
   - centroid_lat: DOUBLE
   - centroid_lng: DOUBLE
   - polygon_wkt: VARCHAR
   - boundary_json: VARCHAR
   - latlng_json: VARCHAR
   - polygon_area: DOUBLE
   - num_vertices: SMALLINT
   - error: VARCHAR
   - created_at: TIMESTAMP
   - h3_derived_id: VARCHAR
   - grid_position_id: VARCHAR
   - primary_address_id: VARCHAR
   - country_code: VARCHAR
   - country_name: VARCHAR
   - state_code: VARCHAR
   - state_name: VARCHAR
   - lga_code: VARCHAR
   - lga_name: VARCHAR
   - ward_code: VARCHAR
   - ward_name: VARCHAR
   - confidence_level: VAR

True

In [50]:
# ALTER COLUMN IF NEEDED 
# Added New addrss columns || Alternatively I can use raw sql
with FastH3DuckDBManager(resolution=7, db_path=H3_DUCKDB_PATH) as db:
    db._add_address_columns()

🚀 Initializing FastH3DuckDBManager (batch_size: 50,000)
📋 Creating optimized h3_cells table...
✅ Optimized table structure created
⚠️  Setting failed: SET wal_autocheckpoint=50000 - Parser Error: Unknown unit for memory_limit: '' (expected: KB, MB, GB, TB for 1000^i units or KiB, MiB, GiB, TiB for 1024^i units)
⚠️  Setting failed: PRAGMA enable_verification=false - Catalog Error: unrecognized configuration parameter "enable_verification"

Did you mean: "enable_profiling"
⚠️  Setting failed: PRAGMA force_checkpoint=false - Catalog Error: unrecognized configuration parameter "force_checkpoint"

Did you mean: "debug_checkpoint_abort"
⚡ Applied 7/10 performance optimizations
✅ Fast H3DuckDBManager ready
Completed adding metadata columns to h3_cells table
🏁 Finalizing...
🎉 Completed: 2,467,165 records in 0.0s
⚡ Average rate: 119,970,321 records/second


In [17]:
import duckdb 
conn = duckdb.connect(H3_DUCKDB_PATH)

In [51]:
# conn.execute("SELECT * FROM h3_cells LIMIT 2").df()

# CHECK MISSING ADDRESS CELLS BY RESOLUTION
conn.execute("""SELECT 
                    resolution, 
                    COUNT(DISTINCT h3_index) total_cells,
                    SUM(CASE WHEN state_code is NULL  
                                  OR lga_code is NULL  
                                  OR ward_code is NULL
                            THEN 1 ELSE 0 END) cell_without_address 
                FROM h3_cells  
                GROUP BY resolution""").fetchdf()  #803,827 # 1,112,246

# CHECK MISSING ADDRESS CELLS
# conn.execute("SELECT count(distinct h3_index) FROM h3_cells where state_code is NULL").fetchdf()  #803,827 # 1,112,246

,resolution,total_cells,cell_without_address
0,8,2158746,803827.0
1,7,308419,308419.0


In [61]:
import pandas as pd
# pd.read_parquet(PATH_DF_H3_ADDRESS_DF_RES7_ALL).head(1)
# pd.read_parquet(PATH_DF_H3_ADDRESS_DF_RES7_FILTER).head(1)
# Address is saved as parquet
# df_address_filtered = duckdb.sql(f"SELECT * EXCLUDE __index_level_0__ FROM '{PATH_DF_H3_ADDRESS_DF_RES7_ALL}'").fetchdf() 
df_address_filtered = duckdb.sql(f"SELECT *  FROM '{PATH_DF_H3_ADDRESS_DF_RES7_ALL}'").fetchdf() 
df_address_filtered.rename(columns={'h3_id':'h3_index'}, inplace=True)
df_address_filtered['resolution'] = 7 
df_address_filtered.head(1)
print(f'Total Records: {len(df_address_filtered):,}')

Total Records: 308,419


In [ ]:
# Set your chunk size
batch_size = 50000
total_processed = 0

# Loop through the DataFrame in chunks
for i in range(0, len(df_address_filtered), batch_size):
    batch_data = df_address_filtered.iloc[i:i + batch_size]
    # print(f"Chunk {i // batch_size + 1}:\n", batch_data.head(), "\n")

    try: 
        conn.register('address_upsert_df', batch_data) 
        # Use INSERT OR REPLACE for UPSERT
        conn.execute("""
            INSERT OR REPLACE INTO h3_cells (
                h3_index, resolution, 
                h3_derived_id, grid_position_id, primary_address_id,
                country_code, country_name, state_code, state_name,
                lga_code, lga_name, ward_code, ward_name,
                confidence_level, coverage_percentage, area_km2
            )
            SELECT 
                h3_index, resolution,
                h3_derived_id, grid_position_id, primary_address_id,
                country_code, country_name, state_code, state_name,
                lga_code, lga_name, ward_code, ward_name,
                confidence_level, coverage_percentage, area_km2
            FROM address_upsert_df
        """)
        
        conn.unregister('address_upsert_df')
        total_processed += len(batch_data)
        
        print(f"✅ Upserted batch {i//batch_size + 1}: {len(batch_data):,} records (total: {total_processed:,})")
        
    except Exception as e:
        print(f"❌ Error upserting : {e}")
        print(f"❌ Error upserting batch {i//batch_size + 1}: {e}")
        continue
    

print(f"🎉 Address data upsert completed: {total_processed:,} records processed")

✅ Upserted batch 1: 50,000 records (total: 50,000)
✅ Upserted batch 2: 50,000 records (total: 100,000)
✅ Upserted batch 3: 50,000 records (total: 150,000)
✅ Upserted batch 4: 50,000 records (total: 200,000)
✅ Upserted batch 5: 50,000 records (total: 250,000)
✅ Upserted batch 6: 50,000 records (total: 300,000)
✅ Upserted batch 7: 8,419 records (total: 308,419)


In [63]:
conn.close()

### 05. Evaluation with Plotting

In [ ]:
# pip install shiny shinywidgets hvplot geoviews geopandas hvplot holoviews bokeh shapely

In [74]:
%load_ext autoreload
%autoreload 2 


import duckdb 
import geopandas as gpd
import pandas as pd
from pathlib import Path
from config.settings import STORAGE_CONFIG 
from src.h3_spatial_system.h3_system.plot_utils import plot_h3_from_db #, plot_h3_from_db_fast

H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']
test_map_path = Path('./output/map/tests')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [66]:
conn = duckdb.connect(H3_DUCKDB_PATH)
# Schema
# conn.execute('DESCRIBE h3_cells;').fetchdf()[['column_name', 'column_type', 'null', 'key']]

In [67]:
df_sample_h3_with_metadata = gpd.GeoDataFrame(conn.execute('SELECT * FROM h3_cells WHERE confidence_level IS NOT NULL LIMIT 100').fetch_df())
# print(df_samole_h3_with_metadata.columns)
df_sample_h3_with_metadata.sample(2)

# ['h3_index', 'resolution', 'centroid_lat', 'centroid_lng', 'polygon_wkt',
# 'boundary_json', 'latlng_json', 'polygon_area', 'num_vertices', 'error',
# 'created_at', 'h3_derived_id', 'grid_position_id', 'primary_address_id',
# 'country_code', 'country_name', 'state_code', 'state_name', 'lga_code',
# 'lga_name', 'ward_code', 'ward_name', 'confidence_level',
# 'coverage_percentage', 'area_km2']

,h3_index,resolution,centroid_lat,centroid_lng,polygon_wkt,boundary_json,latlng_json,polygon_area,num_vertices,error,...,country_name,state_code,state_name,lga_code,lga_name,ward_code,ward_name,confidence_level,coverage_percentage,area_km2
14,88581ea319fffff,8,11.668647,6.353045,POLYGON ((6.349548045405026 11.672084873049952...,"[[11.672084873049952, 6.349548045405026], [11....","[[11.672084873049952, 6.349548045405026], [11....",0.000058,7,None,...,Nigeria,ZA,Zamfara,810,Maru,81001,Bindin,confident,100.0,0.698354
88,8858ee1259fffff,8,9.837928,11.588104,"POLYGON ((11.58468606016931 9.84144750980257, ...","[[9.84144750980257, 11.58468606016931], [9.837...","[[9.84144750980257, 11.58468606016931], [9.837...",0.000059,7,None,...,Nigeria,GO,Gombe,16002,Balanga,GMSBLG04,Degri,confident,100.0,0.720127


In [99]:
# print(conn.execute("SELECT DISTINCT confidence_level FROM h3_cells WHERE state_name = 'Lagos' AND confidence_level IS NOT NULL").fetch_df())
conn.execute("SELECT * FROM h3_cells WHERE h3_derived_id = 'NG-LA-XX-XX-TITMCXR' and lga_name <> 'Unknown' ").fetch_df()
# boundary_case, confident


# Example H3 cell IDs
h3_cells_to_plot =conn.execute("""SELECT DISTINCT h3_index FROM h3_cells 
                               WHERE resolution = 7 
                               AND state_name = 'Lagos' 
                               AND lga_name <> 'Unknown'
                               --AND confidence_level  = 'confident'
                               ---AND lga_name = 'Apapa'
                              --- AND ward_name = 'Abraham Adesanya'
                               """).fetch_df().h3_index.to_list() 

print(len(h3_cells_to_plot))

1077


##### 1. Ploting with folium

In [100]:
plot_folium = plot_h3_from_db(h3_cells_to_plot, H3_DUCKDB_PATH, show_markers=False, 
                popup_fields=['h3_derived_id', 'grid_position_id','state_name', 
                              'lga_name', 'ward_name', 'confidence_level', 'coverage_percentage'
                              #,'centroid_lat', 'centroid_lng'
                              ],
                colors =  ['#3388ff']*len(h3_cells_to_plot),
                # colors = None,
                polygon_weight = 1,
                polygon_opacity = 0.05
                )

# plot_folium

In [101]:
plot_folium.save(test_map_path / 'test_save_as_folium_r7_lagos.html')


#### 4. Using hvploting

In [ ]:
%load_ext autoreload
%autoreload 2 


import duckdb 
import geopandas as gpd
import pandas as pd
from config.settings import STORAGE_CONFIG 
from src.h3_system.plot_utils_dev import plot_h3_from_db_fast

import holoviews as hv
hv.extension('bokeh')

In [ ]:
plot_hv_bokeh = plot_h3_from_db_fast(
    h3_cell_ids = h3_cells_to_plot,
    duckdb_path = H3_DUCKDB_PATH,
    popup_fields = ['h3_derived_id', 'state_name','lga_name', 'ward_name', 'confidence_level', 'coverage_percentage'],
    color_by_column=None,
    static_fill_color='#3388ff',
    # line_color="#000000",
    show_markers=False,
    show_legend=False,
    fill_opacity=0.2,   
    width=800,
    height=600
)

plot_hv_bokeh

In [ ]:

# Save the plot to an HTML file
hv.save(plot_hv_bokeh, test_map_path/'hv_plot.html', backend='bokeh') 



### 06. SP AND CUSTOMER DATA

1. Creating Coverage Area About MFCs at Resolution 8

In [ ]:
%load_ext autoreload
%autoreload 2 

import sys
from pathlib import Path 
import geopandas as gpd

from src.get_data import DataFetcher, get_processed_data, get_geojson_data
from src.data.preprocess_data import preprocess_sp_location_mapping

In [ ]:
# 1. Fetch data from db and store locally: 
# - `sp_dim.sql` -> `df_sp_dim.feather`: Fetches stock point dimension data.
# - `sp_location_map.sql` -> `df_sp_location_mapping.feather`: Fetches the mapping of stock points to locations.
# - `get_customer_dim.sql` -> `df_customer_dim.feather`: Fetches customer dimension data.
# - `sp_active_customers.sql` -> `df_sp_active_customers.feather`: Fetches data for active customers associated with stock points.

## Fetch Data from DB
fetcher = DataFetcher(logger=logger, input_dir=str(RAW_DATA_DIR), sql_dir="_sql")
results = fetcher.fetch_all()


# ETA: 5mins

In [ ]:
%load_ext autoreload
%autoreload 2 

from src.data.preprocess_data import preprocess_sp_location_mapping, prepare_sp_and_recent_activated_customers
from src.data.preprocess_data import load_and_preprocess_sp_lga_mapping_data

# 2. Preprocess Sp Location Mapping LGA
preprocess_sp_location_mapping(logger=logger)
prepare_sp_and_recent_activated_customers(logger)

### 07. SP COVERAGE AREA AND CUSTOMER ASSIGNEMENT


Run complete pipeline
'''
This will return a dictionary with the following keys: ['territories', 'grid_results', 'assignments', 'optimized_clusters', 'statistics', 'territory_version']
1. territories: A dictionary of stock point territories 
        # Dict[stock_point_id, {  
            'polygon': Union[Polygon, MultiPolygon],  
            'lga_ids': List[str],  
            'is_contiguous': bool,  
            'sub_territories': List[Polygon],  
            'total_area_km2': float,  
            'territory_version': str  
        }]  
  
2. grid_results: A dictionary of grid results for each territory  
        Dict[stock_point_id, {  
                'h3_resolution': int,  
                'h3_cells': Set[str],  
                'clipped_cells': Set[str],   
                'cell_geometries': Dict[str, Polygon],  
                'territory_coverage': float  
        }]  
          
3. assignments: A dictionary of customer assignments to stock points  
        Dict[stock_point_id, assignments_gdf with columns:  
                ['customer_id',   
                'cluster_id', 
                'h3_cell_id', 
                'assignment_confidence', 
                'assignment_tier',
                'geometry']]
        
4. optimized_clusters: A dictionary of optimized clusters for each territory

5. statistics: A dictionary of statistics for each territory

6. territory_version: The version of the territory used in the clustering
'''

In [ ]:
%load_ext autoreload
%autoreload 2 

import pandas as pd
from src.H3SpatialClusterer import H3SpatialClusterer  
from src.get_data import get_processed_data #, get_geojson_data,DataFetcher, 
import pickle
from src.utils import clean_customer_gdf_coordinates
import json
from src.utils import filter_cluster_result_dict
from src.plot_utils import plot_geojson_territory_heatmap

In [ ]:
# 3. Fetch store processed data
# lgas_gdf,  sp_dim_df,  stock_point_lga_map, customers_gdf = get_processed_data()
lgas_gdf, sp_dim_df,  stock_point_lga_map, sp_customers_gdf, recent_customers_gdf  = get_processed_data(logger)

#### Setting Up the Pilot Stock Points

In [ ]:
pilot_2_sps = [1647402,	1647372,	1647108,	1646971,	1647109,	1647033,	
               1646999,	1647391,	1647113,	1647137,	1646991,	1647420,	
               1647141,	1647050,	1647421,	1647436,	1647380             ]
pilot_stock_point_lga_map = stock_point_lga_map[stock_point_lga_map['stock_point_id'].isin(pilot_2_sps)]

#### EDA

In [ ]:
## Add Module src/data/preprocess_data.py
## Preprocess all sp location mapping ----------------------------------------
sp_loc_path = INPUT_BASE_DATA_SOURCES['sp_location_mapping']['local_file_path']
df_sp_location_mapping = pd.read_feather(sp_loc_path)
df_sp_location_mapping.columns = df_sp_location_mapping.columns.str.lower()
df_sp_location_mapping = (df_sp_location_mapping
                            .assign(lga_name_=lambda x: x['lga_name'].str.lower())
                            .query('~lga_name_.str.contains("self|push")', engine='python')
                            .drop(columns=['lga_name_'])
                            .reset_index(drop=True)
                            )

print(df_sp_location_mapping.columns.to_list())
print(len(df_sp_location_mapping))

## Preprocess ng lcda geometric file ----------------------------------------
import geopandas as gpd
lcda_geojson_path = ADMIN_DATA_SOURCES['wards']['standardize_file_path']
lcda_gdf = gpd.read_file(lcda_geojson_path)[[  'state_name', 'state_code','lga_name', 'lga_code','ward_name', 'ward_code']] #shape # (9410, 18)
lcda_gdf.columns = [f'{col}_ng' for col in lcda_gdf.columns]
print(lcda_gdf.columns.to_list())
print(len(lcda_gdf))

# Save to disk
# sp_dim_df.to_excel('./output/base_data_export/sp_dim.xlsx', index=False) 
# df_sp_location_mapping.to_excel('./output/base_data_export/sp_location_mapping.xlsx', index=False) 
# lcda_gdf.to_excel('./output/base_data_export/ng_wards.xlsx', index=False) 

In [ ]:
# help(pd.set_option) 

In [ ]:
# Q1. Table of Stock Point and count of LGAs mapped to them
pd.set_option("display.max_row", None)
pd.set_option("display.max_columns", None)

print(stock_point_lga_map.columns.to_list())
# print("--"*100)
# print(f"Distinct Count of LGA mapped to SPs")
# df_eda_sp_lga_count = (stock_point_lga_map.groupby(['stock_point_id', 'stock_point_name'])
#                         .agg( n_map_lgas = ('lga_id','nunique') )
#                         .sort_values('n_map_lga',ascending=False)
#                         .reset_index() 
#                         )

# print(df_eda_sp_lga_count.n_map_lgas.describe())
# print("--"*100)
# print(df_eda_sp_lga_count[['stock_point_name', 'n_map_lgas']].head(3))

# # 2. How many SP are mapped to same location (lga)
# print("--"*100)
# print(f"# 2. How many SPs were mapped to same location (lga)") 
# df_eda_lga_sp_count = (stock_point_lga_map.groupby(['state_id', 'lga_id', 'state_name', 'lga_name'])
#                         .agg(n_map_sps = ('stock_point_id','nunique') )
#                         .sort_values('n_map_sps',ascending=False)
#                         .reset_index() 
#                         )

# df_eda_sp_lga_map_sp_count = (stock_point_lga_map
#                             .merge(df_eda_lga_sp_count[['state_id', 'lga_id', 'n_map_sps']], on=['state_id', 'lga_id'], how='left')
#                             .sort_values(['n_map_sps','state_id', 'lga_id'], ascending=[False, True, True]))


# print(df_eda_lga_sp_count.n_map_sps.describe())
# print("--"*100)
# print(f'Total Number of SP with same lga mapped to another SP', df_eda_sp_lga_map_sp_count.query('n_map_sps > 1').stock_point_id.nunique(), ' Out of ',df_eda_sp_lga_map_sp_count.stock_point_id.nunique() )
# print(df_eda_lga_sp_count[['state_name', 'lga_name', 'n_map_sps']].head(3))
# print(df_eda_sp_lga_map_sp_count.merge(df_eda_lga_sp_count.iloc[0:1][['state_id','lga_id']])[['stock_point_name','state_name', 'lga_name', 'n_map_sps']])


# --------------------------------------------------------------------------------------------
print("--"*100)
print(f"Evaluating Multiple LGA Mapping to Pilot SPs")
# print(pilot_stock_point_lga_map.columns.to_list())

df_eda_lga_sp_count_pilot = (pilot_stock_point_lga_map
                                    .groupby(['state_id', 'lga_id', 'state_name', 'lga_name'])
                                    .agg(n_map_sps = ('stock_point_id','nunique') )
                                    .sort_values('n_map_sps',ascending=False)
                                    .reset_index() 
                                    )
df_eda_sp_lga_map_sp_count_pilot = (pilot_stock_point_lga_map
                                    .merge(df_eda_lga_sp_count_pilot[['state_id', 'lga_id', 'n_map_sps']], on=['state_id', 'lga_id'], how='left')
                                    .sort_values(['n_map_sps','state_id', 'lga_id'], ascending=[False, True, True])) 
print(df_eda_lga_sp_count_pilot.n_map_sps.describe())
print(df_eda_lga_sp_count_pilot.value_counts('n_map_sps'))
print("--"*100)
print(f'List of SPID with same lga mapped to another SP', df_eda_sp_lga_map_sp_count_pilot.query('n_map_sps > 1').stock_point_id.unique())
print(f'Total Number of SP with same lga mapped to another SP', df_eda_sp_lga_map_sp_count_pilot.query('n_map_sps > 1').stock_point_id.nunique(), ' Out of ',df_eda_sp_lga_map_sp_count_pilot.stock_point_id.nunique() )
print(df_eda_lga_sp_count_pilot[['state_name', 'lga_name','n_map_sps']].drop_duplicates().head(3))
print(df_eda_sp_lga_map_sp_count_pilot.merge(df_eda_lga_sp_count_pilot.iloc[0:2][['state_id','lga_id']])[['stock_point_name','state_name', 'lga_name', 'n_map_sps']])



In [ ]:
df_eda_sp_lga_map_sp_count_pilot

#### Stock Point Coverage Mapping and Clustering

In [ ]:
# Pilot sp list
pilot_sps_lists = list(set(pilot_stock_point_lga_map.stock_point_id) )
stock_point_id = str(pilot_sps_lists[1])
print('Total Pilot SPs', len(pilot_sps_lists))

In [ ]:
print(sp_customers_gdf.shape[0])
print(recent_customers_gdf.shape[0])
print(pilot_stock_point_lga_map .shape[0])

In [ ]:
# 1. Initialize with real dataframes
clusterer = H3SpatialClusterer(
    lga_gdf=lgas_gdf,
    sp_dim_df=sp_dim_df[sp_dim_df['stock_point_id'].isin(pilot_2_sps)], 
    stock_point_lga_map=stock_point_lga_map[stock_point_lga_map['stock_point_id'].isin(pilot_2_sps)], 
    customers_gdf=sp_customers_gdf[sp_customers_gdf['stock_point_id'].isin(pilot_2_sps)]
)

In [ ]:
PILOT_SPS_CLUSTER =  clusterer.process_all_stock_points(territory_version="v1.2")

In [ ]:
PILOT_SP_CLUSTERS_PATH = EXPORTS_DIR /  "PILOT_SP_CLUSTERS.pickle"

# Save Results as pickle file
with open(PILOT_SP_CLUSTERS_PATH, 'wb') as f:
    pickle.dump(PILOT_SPS_CLUSTER, f)

In [ ]:
PILOT_SPS_CLUSTER.keys()
# ['territories', 'grid_results', 'assignments', 'optimized_clusters', 'statistics', 'territory_version']
# ALL_CLUSTER['territories']

In [ ]:
# PILOT_SPS_CLUSTER['optimized_clusters']['1646991'] 
# PILOT_SPS_CLUSTER_FLAT.keys() #['clusters', 'assignments', 'territory_summary', 'territory_cells']

print(PILOT_SPS_CLUSTER_FLAT['clusters'].stock_point_id.nunique())
print(PILOT_SPS_CLUSTER_FLAT['territory_cells'].stock_point_id.nunique())

In [ ]:
from src.utils import filter_cluster_result_dict
filtered_result = filter_cluster_result_dict(PILOT_SPS_CLUSTER, pilot_sps_lists) 

filtered_result.keys()

#### Prep sharable Export File

In [ ]:
# 4. Export for deployment: ETA: 1 Min
PILOT_SPS_CLUSTER_FLAT = clusterer.export_results(PILOT_SPS_CLUSTER, output_format="csv")

# Stock Point Assignment Summary
df_output_sp_coverage_cluster = PILOT_SPS_CLUSTER_FLAT['territory_cells']  
df_output_sp_customer_assignment = PILOT_SPS_CLUSTER_FLAT['assignments']
df_output_ap_coverage_cluster_summary = PILOT_SPS_CLUSTER_FLAT['clusters']

df_output_sp_customer_assignment['stock_point_id'] = df_output_sp_customer_assignment['stock_point_id'].astype(int)
df_output_sp_coverage_cluster['stock_point_id'] = df_output_sp_coverage_cluster['stock_point_id'].astype(int)


sp_assignment_summary = (df_output_sp_customer_assignment
                            .groupby(['stock_point_id','cluster_id'])['customer_id'].count()
                            .reset_index(name='n_customers')
                            .rename({'cluster_id':'cell'}, axis=1) 
                        ) 

# Stock Point Coverage - Assignment Summary
sp_coverage_cluster_and_assignment_summary = (df_output_sp_coverage_cluster
                                            .merge(sp_dim_df[['stock_point_id', 'stock_point_name']] , on='stock_point_id', how='left')
                                            .merge(sp_assignment_summary, how='left', on=['stock_point_id','cell'])
                                            .fillna({'n_customers':0})
                                            .rename({'cell':'cluster_id'}, axis=1)
                                            ) 
sp_coverage_cluster_and_assignment_summary['n_customers'] = sp_coverage_cluster_and_assignment_summary['n_customers'].astype(int) 

 
print(PILOT_SPS_CLUSTER_FLAT['territory_cells'].stock_point_id.nunique())
print(sp_coverage_cluster_and_assignment_summary.stock_point_id.nunique())

In [ ]:
# df_output_assignment.head(10)
# print(df_output_sp_customer_assignment.columns.to_list)
# print(df_output_sp_customer_assignment.assignment_tier.value_counts())
# print(df_output_sp_customer_assignment.assignment_confidence.value_counts())

# df_output_sp_customer_assignment.head(2)
# sp_assignment_summary.head(2)
 

In [ ]:
import duckdb 
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']
conn = duckdb.connect(H3_DUCKDB_PATH)

# Install and load the httpfs extension
# conn.execute("INSTALL 'httpfs';")
conn.execute("LOAD 'httpfs';")

# Load the parquet extension if needed
conn.execute("LOAD 'parquet';")


In [ ]:
## Customer Assignment Table Enhanced
df_output_sp_customer_assignment_enhanced = conn.execute("""SELECT 
                a.stock_point_id, c.stock_point_name, a.customer_id, cluster_id, h3_derived_id cluster_code, 
                ROUND(assignment_confidence * 100, 2) assignment_confidence,
                CASE WHEN assignment_tier = 'h3_inclusion' THEN 'within cluster' 
                    WHEN assignment_tier = 'manual_review' THEN 'manual review'
                ELSE assignment_tier END AS assignment_tier,
                d.business_id,  d.contact_name, d.customer_status, kyc_capture_status, agent_id, agent_name, 
                d.state_name as customer_state_name,	
                d.town_name as customer_town_name,	
                d.city_name as customer_city_name
                FROM df_output_sp_customer_assignment a
                LEFT JOIN h3_cells b ON b.h3_index = a.cluster_id
                LEFT JOIN sp_dim_df c ON c.stock_point_id = a.stock_point_id 
                LEFT JOIN read_parquet('/home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/processed/df_processed_customer_dim.parquet') d ON d.customer_id = a.customer_id             
                -- LIMIT 2
                """).df()

print(PILOT_SPS_CLUSTER_FLAT['assignments'].stock_point_id.nunique())
print(df_output_sp_customer_assignment_enhanced.stock_point_id.nunique())
df_output_sp_customer_assignment_enhanced.head(2)

In [ ]:
print(len(df_output_sp_customer_assignment))
print(len(df_output_sp_customer_assignment_enhanced))

In [ ]:
# print(conn.execute('SELECT * FROM h3_cells LIMIT 1').df().columns.to_list())
# ['h3_index', 'resolution', 'centroid_lat', 'centroid_lng', 'polygon_wkt', 'boundary_json', 
#  'latlng_json', 'polygon_area', 'num_vertices', 'error', 'created_at', 'h3_derived_id', 
#  'grid_position_id', 'primary_address_id', 'country_code', 'country_name', 'state_code', 
#  'state_name', 'lga_code', 'lga_name', 'ward_code', 'ward_name', 'confidence_level', 
#  'coverage_percentage', 'area_km2']
from src.utils import calculate_distance_km
sp_coverage_cluster_and_assignment_summary_enhanced = conn.execute('''SELECT 
                    a.stock_point_id, a.stock_point_name,
                    a.cluster_id, h3_derived_id as cluster_code, 
                    a.n_customers as customer_count,
                    confidence_level as cluster_address_level, 
                    state_name as cluster_state_name, lga_name as cluster_lga_name, ward_name as cluster_ward_name,
                    --- Add coord columns if needed,
                    centroid_lat as cluster_centroid_lat, 
                    centroid_lng as cluster_centroid_lng,
                    latitude as sp_lat, 
                    longitude as sp_lng
                FROM sp_coverage_cluster_and_assignment_summary a
                LEFT JOIN  sp_dim_df b ON b.stock_point_id = a.stock_point_id
                LEFT JOIN h3_cells b ON b.h3_index = a.cluster_id 
            ''').df()

sp_coverage_cluster_and_assignment_summary_enhanced['cluster_sp_dist_km'] = (sp_coverage_cluster_and_assignment_summary_enhanced
                                                                             .apply(lambda row: calculate_distance_km(row['cluster_centroid_lat'], row['cluster_centroid_lng'], 
                                                                                                                      row['sp_lat'], row['sp_lng']), axis=1)
                                                                            )
get_cluster_status = lambda dist: 'Undefined' if dist is None else 'Within 7km' if dist <= 7 else 'Above 7km'
sp_coverage_cluster_and_assignment_summary_enhanced['cluster_sp_dist_status'] = (sp_coverage_cluster_and_assignment_summary_enhanced['cluster_sp_dist_km']
                                                                             .apply(lambda x: get_cluster_status(x))
                                                                            )    
try:
    drp_cols = [ 'cluster_centroid_lat','cluster_centroid_lng', 'sp_lat', 'sp_lng']
    sp_coverage_cluster_and_assignment_summary_enhanced.drop(drp_cols, axis=1, inplace=True)
except Exception as e:
    print(e) 


print(PILOT_SPS_CLUSTER_FLAT['territory_cells'].stock_point_id.nunique())
print(sp_coverage_cluster_and_assignment_summary.stock_point_id.nunique())
print(sp_coverage_cluster_and_assignment_summary_enhanced.stock_point_id.nunique())

sp_coverage_cluster_and_assignment_summary_enhanced.sample(2) 

In [ ]:
# Create a Pandas Excel writer using openpyxl as the engine
with pd.ExcelWriter(OUTPUT_DIR / 'pilot 2/coverage_cluster_and_customer_assignment.xlsx', engine='openpyxl') as writer: 
    sp_coverage_cluster_and_assignment_summary_enhanced.to_excel(writer, sheet_name='coverage_cluster', index=False)
    df_output_sp_customer_assignment_enhanced.to_excel(writer, sheet_name='assignment', index=False)